<a href="https://colab.research.google.com/github/miriam-silva/MineiracaoDeDados/blob/main/ETLCerto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np
import re
import os

from google.colab import files

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [34]:
# Carregamento da base de dados

print("Selecione o arquivo CSV da base de dados.")

uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

print(f"\nArquivo carregado: {arquivo}")

Selecione o arquivo CSV da base de dados.


Saving manifestacoes-de-ouvidoria-ouverj(Ouvidoria).csv to manifestacoes-de-ouvidoria-ouverj(Ouvidoria) (1).csv

Arquivo carregado: manifestacoes-de-ouvidoria-ouverj(Ouvidoria) (1).csv


In [35]:
# Extração

df_bruto = pd.read_csv(
    arquivo,
    sep=';',
    encoding='utf-8-sig',
    dtype=str
)

print("Base extraída com sucesso!")
print(f"Quantidade de registros: {df_bruto.shape[0]:,}")
print(f"Quantidade de colunas: {df_bruto.shape[1]}")

Base extraída com sucesso!
Quantidade de registros: 134,217
Quantidade de colunas: 9


In [36]:
print("Dimensões da base:")
print(df_bruto.shape)

print("\nColunas:")
for coluna in df_bruto.columns:
    print("-", coluna)

print("\nPrimeiros registros:")
display(df_bruto.head())

Dimensões da base:
(134217, 9)

Colunas:
- Protocolo
- Assunto
- Criado em
- Status
- Órgão
- Gênero
- Faixa Etária
- Escolaridade
- Município

Primeiros registros:


,Protocolo,Assunto,Criado em,Status,Órgão,Gênero,Faixa Etária,Escolaridade,Município
0,202506092669316,Arquivo Público,08/06/2025,Em Aberto,Secretaria de Estado da Casa Civil,-,80 anos ou mais,Não Informado,Rio de Janeiro
1,202506082557219,Atendimento,08/06/2025,Concluído,Secretaria de Estado de Polícia Civil,-,80 anos ou mais,Não Informado,-
2,202506082106813,Atendimento,08/06/2025,Concluído,Secretaria de Estado de Polícia Civil,-,-,-,-
3,202506082264867,Atendimento,08/06/2025,Concluído,Secretaria de Estado de Polícia Civil,-,-,-,-
4,202506082347885,Atendimento,08/06/2025,Concluído,Secretaria de Estado de Polícia Civil,-,-,-,-


In [37]:
print("Tipos de dados inicialmente:")
print(df_bruto.dtypes)

Tipos de dados inicialmente:
Protocolo       object
Assunto         object
Criado em       object
Status          object
Órgão           object
Gênero          object
Faixa Etária    object
Escolaridade    object
Município       object
dtype: object


In [38]:
diagnostico_nulos = pd.DataFrame({
    'Coluna': df_bruto.columns,
    'Valores_Nulos': df_bruto.isna().sum().values,
    'Percentual_Nulos': (
        df_bruto.isna().mean().values * 100
    ).round(2)
})

display(diagnostico_nulos)

,Coluna,Valores_Nulos,Percentual_Nulos
0,Protocolo,0,0.00
1,Assunto,0,0.00
2,Criado em,0,0.00
3,Status,0,0.00
4,Órgão,0,0.00
5,Gênero,21,0.02
6,Faixa Etária,0,0.00
7,Escolaridade,26,0.02
8,Município,0,0.00


In [39]:
contagem_tracos = pd.DataFrame({
    'Coluna': df_bruto.columns,
    'Quantidade_de_-': [
        (df_bruto[coluna] == '-').sum()
        for coluna in df_bruto.columns
    ]
})

display(contagem_tracos)

,Coluna,Quantidade_de_-
0,Protocolo,201
1,Assunto,4201
2,Criado em,201
3,Status,201
4,Órgão,201
5,Gênero,98021
6,Faixa Etária,70286
7,Escolaridade,70287
8,Município,38780


In [40]:
linhas_sem_protocolo = df_bruto[df_bruto['Protocolo'] == '-']

print(
    f"Registros com protocolo '-': "
    f"{len(linhas_sem_protocolo):,}"
)

Registros com protocolo '-': 201


In [41]:
indice_anomalia = df_bruto.index[
    df_bruto['Protocolo'] == '-'
][0]

print(f"A primeira linha da anomalia ocorre no índice: {indice_anomalia}")

print("\nÚltimo registro válido antes da anomalia:")
display(df_bruto.iloc[[indice_anomalia - 1]])

print("\nPrimeiro registro da anomalia:")
display(df_bruto.iloc[[indice_anomalia]])

A primeira linha da anomalia ocorre no índice: 134016

Último registro válido antes da anomalia:


,Protocolo,Assunto,Criado em,Status,Órgão,Gênero,Faixa Etária,Escolaridade,Município
134015,2023111352132,Gestão,13/11/2023,Concluído,Centro de Tecnologia de Informação e Comunicaç...,-,80 anos ou mais,Não Informado,-



Primeiro registro da anomalia:


,Protocolo,Assunto,Criado em,Status,Órgão,Gênero,Faixa Etária,Escolaridade,Município
134016,-,-,-,-,-,NaN,80 anos ou mais,NaN,-


In [42]:
# Transformação

In [43]:
df = df_bruto.copy()

print("Cópia da base criada para transformação.")
print(f"Registros antes da limpeza: {len(df):,}")

Cópia da base criada para transformação.
Registros antes da limpeza: 134,217


In [44]:
quantidade_antes = len(df)

df = df[df['Protocolo'] != '-'].copy()

quantidade_depois = len(df)

removidos = quantidade_antes - quantidade_depois

print(f"Registros antes: {quantidade_antes:,}")
print(f"Registros removidos: {removidos:,}")
print(f"Registros após a remoção: {quantidade_depois:,}")

Registros antes: 134,217
Registros removidos: 201
Registros após a remoção: 134,016


In [45]:
print("Registros com protocolo '-' após a limpeza:")
print((df['Protocolo'] == '-').sum())

Registros com protocolo '-' após a limpeza:
0


In [46]:
colunas_texto = [
    'Assunto',
    'Status',
    'Órgão',
    'Gênero',
    'Faixa Etária',
    'Escolaridade',
    'Município'
]

for coluna in colunas_texto:
    df[coluna] = (
        df[coluna]
        .fillna('')
        .astype(str)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

print("Espaços extras removidos dos campos textuais.")

Espaços extras removidos dos campos textuais.


In [47]:
for coluna in colunas_texto:
    quantidade_vazia = (
        df[coluna].isna().sum()
    )

    print(
        f"{coluna}: "
        f"{quantidade_vazia} valores nulos"
    )

Assunto: 0 valores nulos
Status: 0 valores nulos
Órgão: 0 valores nulos
Gênero: 0 valores nulos
Faixa Etária: 0 valores nulos
Escolaridade: 0 valores nulos
Município: 0 valores nulos


In [48]:
for coluna in df.columns:
    df[coluna] = df[coluna].replace(
        ['-', ''],
        'Não Informado'
    )

print("Valores ausentes padronizados como 'Não Informado'.")

Valores ausentes padronizados como 'Não Informado'.


In [49]:
print("Quantidade de '-' restante na base:")

for coluna in df.columns:
    quantidade = (df[coluna] == '-').sum()

    if quantidade > 0:
        print(f"{coluna}: {quantidade}")

Quantidade de '-' restante na base:


In [50]:
def limpar_texto(texto):
    if pd.isna(texto) or str(texto).strip() == '':
        return 'Não Informado'

    texto = str(texto).strip()
    texto = re.sub(r'\s+', ' ', texto)

    if texto == '-':
        return 'Não Informado'

    return texto


for coluna in colunas_texto:
    df[coluna] = df[coluna].apply(limpar_texto)

print("Higienização textual concluída.")

Higienização textual concluída.


In [51]:
for coluna in colunas_texto:
    df[coluna] = df[coluna].apply(padronizar_texto)

print("Normalização textual concluída.")

Normalização textual concluída.


In [52]:
df['Criado em'] = pd.to_datetime(
    df['Criado em'],
    format='%d/%m/%Y',
    errors='coerce'
)

print("Conversão da coluna 'Criado em' concluída.")
print(df['Criado em'].dtype)

Conversão da coluna 'Criado em' concluída.
datetime64[ns]


In [53]:
datas_invalidas = df['Criado em'].isna().sum()

print(
    f"Quantidade de datas inválidas ou ausentes: "
    f"{datas_invalidas}"
)

Quantidade de datas inválidas ou ausentes: 0


In [54]:
print(
    "Data mais antiga:",
    df['Criado em'].min().strftime('%d/%m/%Y')
)

print(
    "Data mais recente:",
    df['Criado em'].max().strftime('%d/%m/%Y')
)

Data mais antiga: 13/11/2023
Data mais recente: 08/06/2025


In [55]:
duplicados_protocolo = df['Protocolo'].duplicated().sum()

print(
    f"Protocolos duplicados: "
    f"{duplicados_protocolo}"
)

Protocolos duplicados: 0


In [56]:
duplicados_linhas = df.duplicated().sum()

print(
    f"Registros completamente duplicados: "
    f"{duplicados_linhas}"
)

Registros completamente duplicados: 0


In [57]:
diagnostico_final = pd.DataFrame({
    'Coluna': df.columns,
    'Nulos': df.isna().sum().values,
    'Percentual_Nulos': (
        df.isna().mean().values * 100
    ).round(2)
})

display(diagnostico_final)

,Coluna,Nulos,Percentual_Nulos
0,Protocolo,0,0.0
1,Assunto,0,0.0
2,Criado em,0,0.0
3,Status,0,0.0
4,Órgão,0,0.0
5,Gênero,0,0.0
6,Faixa Etária,0,0.0
7,Escolaridade,0,0.0
8,Município,0,0.0


In [58]:
for coluna in df.columns:
    print(f"\n===== {coluna} =====")
    print(df[coluna].value_counts().head(10))


===== Protocolo =====
Protocolo
2023120157909    1
2023120138819    1
2023120139869    1
2023120137819    1
2023120136874    1
2023120135533    1
2023120134123    1
2023120132498    1
2023120132078    1
2023120131295    1
Name: count, dtype: int64

===== Assunto =====
Assunto
Atendimento                23478
Veículos                   14490
Habilitação                13259
Identificação Civil        11611
Diplomas E Certificados     6567
Taxas                       6165
Multas                      4485
Transporte                  4362
Não Informado               4000
Gestão                      2998
Name: count, dtype: int64

===== Criado em =====
Criado em
2023-12-19    1215
2023-12-18     764
2024-06-25     631
2024-06-26     585
2024-01-23     574
2025-04-28     568
2025-02-19     544
2025-04-30     520
2025-03-18     511
2025-05-20     507
Name: count, dtype: int64

===== Status =====
Status
Concluído                 116577
Arquivado                  14781
Em Andamento            

In [59]:
print("SCHEMA FINAL DA BASE")
print("=" * 40)

for coluna in df.columns:
    print(f"{coluna}: {df[coluna].dtype}")

SCHEMA FINAL DA BASE
Protocolo: object
Assunto: object
Criado em: datetime64[ns]
Status: object
Órgão: object
Gênero: object
Faixa Etária: object
Escolaridade: object
Município: object


In [60]:
print("Dimensões finais:")
print(df.shape)

print("\nPrimeiros registros tratados:")
display(df.head())

print("\nÚltimos registros tratados:")
display(df.tail())

Dimensões finais:
(134016, 9)

Primeiros registros tratados:


,Protocolo,Assunto,Criado em,Status,Órgão,Gênero,Faixa Etária,Escolaridade,Município
0,202506092669316,Arquivo Público,2025-06-08,Em Aberto,Secretaria De Estado Da Casa Civil,Não Informado,80 Anos Ou Mais,Não Informado,Rio De Janeiro
1,202506082557219,Atendimento,2025-06-08,Concluído,Secretaria De Estado De Polícia Civil,Não Informado,80 Anos Ou Mais,Não Informado,Não Informado
2,202506082106813,Atendimento,2025-06-08,Concluído,Secretaria De Estado De Polícia Civil,Não Informado,Não Informado,Não Informado,Não Informado
3,202506082264867,Atendimento,2025-06-08,Concluído,Secretaria De Estado De Polícia Civil,Não Informado,Não Informado,Não Informado,Não Informado
4,202506082347885,Atendimento,2025-06-08,Concluído,Secretaria De Estado De Polícia Civil,Não Informado,Não Informado,Não Informado,Não Informado



Últimos registros tratados:


,Protocolo,Assunto,Criado em,Status,Órgão,Gênero,Faixa Etária,Escolaridade,Município
134011,2023113033730,Taxas,2023-11-30,Concluído,Departamento De Trânsito Do Estado Do Rio De J...,Não Informado,Não Informado,Não Informado,Não Informado
134012,2023113031795,Veículos,2023-11-30,Arquivado,Departamento De Trânsito Do Estado Do Rio De J...,Não Informado,Não Informado,Não Informado,Não Informado
134013,2023113031325,Veículos,2023-11-30,Concluído,Departamento De Trânsito Do Estado Do Rio De J...,Não Informado,Não Informado,Não Informado,Não Informado
134014,2023112959724,Bens Móveis,2023-11-29,Concluído,Secretaria De Estado De Planejamento E Gestão,Não Informado,80 Anos Ou Mais,Não Informado,Não Informado
134015,2023111352132,Gestão,2023-11-13,Concluído,Centro De Tecnologia De Informação E Comunicaç...,Não Informado,80 Anos Ou Mais,Não Informado,Não Informado


In [61]:
print("VALIDAÇÃO FINAL DO ETL")
print("=" * 50)

print(f"Registros brutos:              {len(df_bruto):,}")
print(f"Registros removidos:           {len(df_bruto) - len(df):,}")
print(f"Registros finais:              {len(df):,}")
print(f"Colunas finais:                {len(df.columns)}")
print(f"Protocolos duplicados:         {df['Protocolo'].duplicated().sum():,}")
print(f"Linhas duplicadas:             {df.duplicated().sum():,}")
print(f"Datas inválidas:               {df['Criado em'].isna().sum():,}")
print(f"Valores '-' restantes:         {(df == '-').sum().sum():,}")

print("\nSchema:")
print(df.dtypes)

VALIDAÇÃO FINAL DO ETL
Registros brutos:              134,217
Registros removidos:           201
Registros finais:              134,016
Colunas finais:                9
Protocolos duplicados:         0
Linhas duplicadas:             0
Datas inválidas:               0
Valores '-' restantes:         0

Schema:
Protocolo               object
Assunto                 object
Criado em       datetime64[ns]
Status                  object
Órgão                   object
Gênero                  object
Faixa Etária            object
Escolaridade            object
Município               object
dtype: object


In [62]:
arquivo_saida = 'manifestacoes_ouverj_tratado.csv'

df.to_csv(
    arquivo_saida,
    sep=';',
    index=False,
    encoding='utf-8-sig'
)

print(f"Arquivo final criado: {arquivo_saida}")

Arquivo final criado: manifestacoes_ouverj_tratado.csv


In [63]:
files.download(arquivo_saida)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [64]:
print("""
===============================================
        RELATÓRIO FINAL DO PROCESSO ETL
===============================================
""")

print("FERRAMENTA")
print("Python + Pandas / Google Colab")

print("\nFONTE")
print("Portal de Dados Abertos do Governo")

print("\nDATASET")
print("Manifestações de Ouvidoria de 2025 - OuvERJ")

print("\nVOLUME")
print(f"Registros brutos: {len(df_bruto):,}")
print(f"Registros finais: {len(df):,}")

print("\nTRANSFORMAÇÕES")
print("- Remoção de registros estruturalmente inconsistentes")
print("- Padronização de valores ausentes")
print("- Remoção de espaços extras")
print("- Normalização dos textos")
print("- Conversão da data para datetime")
print("- Validação de duplicidades")
print("- Validação de datas")

print("\nRESULTADO")
print(f"Registros removidos: {len(df_bruto) - len(df):,}")
print(f"Registros carregados: {len(df):,}")
print(f"Colunas finais: {len(df.columns)}")
print(f"Protocolos duplicados: {df['Protocolo'].duplicated().sum():,}")
print(f"Linhas duplicadas: {df.duplicated().sum():,}")
print(f"Datas inválidas: {df['Criado em'].isna().sum():,}")

print("\n===============================================")


        RELATÓRIO FINAL DO PROCESSO ETL

FERRAMENTA
Python + Pandas / Google Colab

FONTE
Portal de Dados Abertos do Governo

DATASET
Manifestações de Ouvidoria de 2025 - OuvERJ

VOLUME
Registros brutos: 134,217
Registros finais: 134,016

TRANSFORMAÇÕES
- Remoção de registros estruturalmente inconsistentes
- Padronização de valores ausentes
- Remoção de espaços extras
- Normalização dos textos
- Conversão da data para datetime
- Validação de duplicidades
- Validação de datas

RESULTADO
Registros removidos: 201
Registros carregados: 134,016
Colunas finais: 9
Protocolos duplicados: 0
Linhas duplicadas: 0
Datas inválidas: 0

